# Question 1 Evaluation Results — Applied-Criteria-Only Analysis

## 1. Motivation

The "Updated Evaluation" in the original case study (`case_study.ipynb`) scored every (variant, criterion) pair the pipeline touched across all 200 variants, treating any criterion not explicitly listed in `gt_criteria_applied` as a gold negative. This fixed the original script's problem of too few negative examples, but it introduced a new imbalance: most of the 200 variants in `full_evaluation_dataset.csv` have **no** curator-specified applied criteria at all (only 48 of 200 have a non-empty `gt_criteria_applied`). For a criterion like `PM2_SUPPORTING`, that meant N jumped from 34 (original script) to 200 (updated script), with the extra ~166 pairs being "confirmed negatives" contributed almost entirely by variants the curator never actually evaluated that criterion against.

This case study reruns Question 1 restricted to the **48 variants that have at least one applied criterion**, to see how much of the "Updated Evaluation" results were driven by that imbalance versus genuine model weaknesses. The gold-label logic within `question1.py` is otherwise unchanged: for the 48 variants kept, a pipeline-evaluated criterion still counts as gold-negative if it isn't listed in that variant's `gt_criteria_applied`.

Command used:

```bash
python -m evaluation.question1 \
  --gold_answer_csv_filename applied_only_evaluation_dataset.csv \
  --model_output_json_filename outputs/batch_20260708_132813/batch_summary.json \
  --output_dir evaluation/outputs
```

where `applied_only_evaluation_dataset.csv` is `full_evaluation_dataset.csv` filtered to rows with a non-empty `gt_criteria_applied`, and `batch_20260708_132813` is the same 200-variant pipeline run used in the original case study.


## 2. Applied-Only Evaluation Results (48 variants)

Overall: accuracy 0.943, precision 0.777, recall 0.967, F1 0.861, across 487 scored (variant, criterion) pairs. 1,347 predicted criteria from the excluded 152 variants were dropped as "unmatched" (expected, since those variants are no longer in the gold set).

| Criterion | N | Accuracy | Precision | Recall | F1 |
|-----------|---|----------|-----------|--------|----|
| BA1 | 48 | 1.000 | 1.000 | 1.000 | 1.000 |
| BP4 | 40 | 0.975 | 0.800 | 1.000 | 0.889 |
| BS1 | 48 | 0.917 | 0.600 | 1.000 | 0.750 |
| BS3 | 48 | 1.000 | 0.000 | 0.000 | 0.000 |
| PM1 | 38 | 1.000 | 1.000 | 1.000 | 1.000 |
| PM2_SUPPORTING | 48 | 1.000 | 1.000 | 1.000 | 1.000 |
| PM4 | 1 | 1.000 | 1.000 | 1.000 | 1.000 |
| PM5 | 38 | 0.789 | 0.333 | 1.000 | 0.500 |
| PP3 | 40 | 0.975 | 1.000 | 0.941 | 0.970 |
| PS1 | 38 | 0.974 | 0.000 | 0.000 | 0.000 |
| PS3 | 48 | 0.979 | 0.000 | 0.000 | 0.000 |
| PS4 | 47 | 0.787 | 0.583 | 1.000 | 0.737 |
| PVS1 | 5 | 0.600 | 0.600 | 1.000 | 0.750 |


## 3. Comparison to the Full 200-Variant ("Updated") Evaluation

For reference, the full-200-variant results from the original case study:

| Criterion | N | Accuracy | Precision | Recall | F1 |
|-----------|---|----------|-----------|--------|----|
| BA1 | 199 | 1.000 | 1.000 | 1.000 | 1.000 |
| BP4 | 138 | 0.659 | 0.078 | 1.000 | 0.145 |
| BP7 | 33 | 0.030 | 0.000 | 0.000 | 0.000 |
| BS1 | 199 | 0.980 | 0.600 | 1.000 | 0.750 |
| BS3 | 200 | 1.000 | 0.000 | 0.000 | 0.000 |
| PM1 | 90 | 0.978 | 0.600 | 1.000 | 0.750 |
| PM2_SUPPORTING | 200 | 0.250 | 0.185 | 1.000 | 0.312 |
| PM4 | 3 | 0.333 | 0.333 | 1.000 | 0.500 |
| PM5 | 90 | 0.711 | 0.133 | 1.000 | 0.235 |
| PP3 | 138 | 0.833 | 0.421 | 0.941 | 0.582 |
| PS1 | 90 | 0.989 | 0.000 | 0.000 | 0.000 |
| PS3 | 200 | 0.995 | 0.000 | 0.000 | 0.000 |
| PS4 | 199 | 0.894 | 0.400 | 1.000 | 0.571 |
| PVS1 | 53 | 0.057 | 0.057 | 1.000 | 0.107 |

Side-by-side, three patterns stand out:

**PM2_SUPPORTING flips completely.** In the full-200 run it was one of the worst performers (accuracy 0.250, F1 0.312) — but restricted to the 48 applied-only variants, it is perfect (accuracy 1.000, F1 1.000, N=48). This strongly suggests the full-200 score was an artifact of exactly the imbalance described in Section 1: the model applies PM2_SUPPORTING correctly whenever there is a real applied/not-applied decision to make, but the full evaluation penalized it against ~150 variants whose gold label was defaulted to "not applied" without a curator ever weighing in on that criterion for them.

**BP4 and PVS1 improve substantially but remain imperfect.** BP4 goes from accuracy 0.659/F1 0.145 to 0.975/0.889; PVS1 goes from 0.057/0.107 to 0.600/0.750. Both are much healthier once the defaulted-negative noise is removed, but PVS1 in particular is still weak (N=5, so this is a very small sample) and BP4's precision, while much better, still isn't perfect.

**BS3, PS1, and PS3 do not improve at all.** All three still show precision = 0, recall = 0, F1 = 0 in the applied-only subset (BS3: N=48, PS1: N=38, PS3: N=48), just as in the full-200 run. Because this happens even after removing the defaulted-negative variants, it points to a genuine model weakness (or a genuine absence of true positives for these criteria in the sample) rather than an artifact of how gold negatives are assigned.


## 4. Interpretation

Filtering to applied-only variants isolates the model's real per-criterion competence from the sample-imbalance problem. The main takeaway is that **PM2_SUPPORTING was never actually a weak criterion** — the full-dataset evaluation was measuring something closer to "how often does the model correctly output False for a criterion no one asked about," which happened to be a bad match for how liberally PM2_SUPPORTING is applied.

The criteria that remain weak after filtering (BS3, PS1, PS3, and to a lesser extent BP4/PVS1) are better candidates for follow-up error analysis in the style of the original case study's BP7 investigation (Section 5 of `case_study.ipynb`) — pulling the individual predictions for these criteria and comparing evidence/reasoning against gold labels to determine whether the issue is missing exceptions in the instructions, insufficient evidence gathering, or judge-agent reasoning errors.

## 5. Caveats

- This analysis still treats any criterion not listed in a kept variant's `gt_criteria_applied` as a gold negative — it narrows the population to variants with at least one applied criterion, but it does not change that underlying assumption for the criteria not mentioned by those 48 variants.
- Several criteria (PM4: N=1, PVS1: N=5) have too few examples in the applied-only subset for their accuracy/precision/recall/F1 to be statistically meaningful, the same caveat raised for PS1 in the original case study.
- The 1,347 "unmatched" predictions are pipeline output from the 152 excluded variants — they are not evidence of pipeline error, just predictions with no corresponding row in the filtered gold CSV.
